# LLM Providers

From the [LLMs documentation](../../../docs/source/workflows/llms/index.md):

## Supported LLM Providers

NeMo Agent Toolkit supports the following LLM providers:

| Provider | Type | Documentation |
|----------|------|---------------|
| **NVIDIA NIM** | `nim` | [NVIDIA NIM](https://build.nvidia.com) |
| **OpenAI** | `openai` | [OpenAI API](https://openai.com) |
| **AWS Bedrock** | `aws_bedrock` | [AWS Bedrock](https://aws.amazon.com/bedrock/) |
| **Azure OpenAI** | `azure_openai` | [Azure OpenAI](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/quickstart) |
| **LiteLLM** | `litellm` | [LiteLLM](https://github.com/BerriAI/litellm) |

## What You'll Learn

1. NVIDIA NIM LLM configuration
2. OpenAI LLM configuration
3. Common LLM parameters
4. Switching between providers

For the full list of providers and options, see the [LLMs documentation](../../../docs/source/workflows/llms/index.md).


In [1]:
import getpass
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

# Load environment variables from .env file
load_dotenv()

# Check for NVIDIA API key (required for NIM)
nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

if nvidia_api_key:
    print("✅ NVIDIA_API_KEY loaded")
else:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA_API_KEY (get one at https://build.nvidia.com/): ")
    if nvidia_api_key:
        os.environ["NVIDIA_API_KEY"] = nvidia_api_key
        print("✅ NVIDIA_API_KEY set")
    else:
        print("⚠️ NVIDIA_API_KEY not set - NIM examples will fail")


✅ NVIDIA_API_KEY loaded


## 1. NVIDIA NIM LLM

NVIDIA NIM provides access to various models including Llama, Mistral, and more:


In [2]:
from nat.llm.sdk import NimLLM

# Basic NIM configuration
nim_llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,        # 0.0 = deterministic, 1.0 = creative
    max_tokens=1024,        # Maximum response length
    name="nim_llm",
)

print(f"✅ NIM LLM: {nim_llm.model_name}")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ NIM LLM: meta/llama-3.3-70b-instruct


In [3]:
# NIM with advanced options
nim_advanced = NimLLM(
    model_name="meta/llama-3.1-70b-instruct",
    temperature=0.7,
    max_tokens=2048,
    top_p=0.9,              # Nucleus sampling
    name="nim_advanced",
)

print(f"✅ Advanced NIM: {nim_advanced.model_name}")


✅ Advanced NIM: meta/llama-3.1-70b-instruct


### Popular NIM Models

| Model | Name | Best For |
|-------|------|----------|
| Llama 3.3 70B | `meta/llama-3.3-70b-instruct` | General purpose, high quality |
| Llama 3.1 70B | `meta/llama-3.1-70b-instruct` | General purpose |
| Llama 3.1 8B | `meta/llama-3.1-8b-instruct` | Fast, cost-effective |
| Mistral Large | `mistralai/mistral-large-latest` | Strong reasoning |


## 2. OpenAI LLM

For OpenAI models, you need an API key. We use `python-dotenv` to load it from a `.env` file:


In [4]:
# Check for OpenAI API key (optional, for OpenAI examples)
openai_api_key = os.environ.get("OPENAI_API_KEY")

if openai_api_key:
    print("✅ OPENAI_API_KEY loaded")
else:
    openai_api_key = getpass.getpass("Enter your OPENAI_API_KEY (or press Enter to skip): ")
    if openai_api_key:
        os.environ["OPENAI_API_KEY"] = openai_api_key
        print("✅ OPENAI_API_KEY set")
    else:
        print("⏭️ Skipping OpenAI examples")


✅ OPENAI_API_KEY loaded


In [5]:
from nat.llm.sdk import OpenAILLM
from nat.utils.sdk.nat_env_var import NatEnvironmentVariable

# Create OpenAI LLM only if API key is available
openai_llm = None

if openai_api_key:
    openai_llm = OpenAILLM(
        model_name="gpt-4o",
        temperature=0.0,
        api_key_env=NatEnvironmentVariable(name="OPENAI_API_KEY"),
        name="openai_llm",
    )
    print(f"✅ OpenAI LLM: {openai_llm.model_name}")
else:
    print("⏭️ Skipping OpenAI LLM creation (no API key)")

✅ OpenAI LLM: gpt-4o


### Popular OpenAI Models

| Model | Name | Best For |
|-------|------|----------|
| GPT-4o | `gpt-4o` | Best overall quality |
| GPT-4o Mini | `gpt-4o-mini` | Fast, cost-effective |
| GPT-4 Turbo | `gpt-4-turbo` | Complex reasoning |


## 3. Test NIM LLM


In [6]:
from nat.agent.sdk import NatReActAgent
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_workflow import NatWorkflow

time_tool = CurrentTimeTool(name="current_time")

# Create agent with NIM LLM
nim_agent = NatReActAgent(
    tools=[time_tool],
    llm=nim_llm,
    verbose=True,
)

nim_workflow = NatWorkflow(entrypoint=nim_agent)
print("✅ NIM Workflow created")


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ NIM Workflow created


In [7]:
# Test the NIM workflow
result = await nim_workflow.prompt("What time is it right now?")
print(f"\n🤖 NIM Response:\n{result}")



🤖 NIM Response:
The current time is 2025-12-30 00:15:29 +0000.


## 4. Test OpenAI LLM


In [8]:
# Create and test OpenAI workflow (only if API key is available)
if openai_llm:
    openai_agent = NatReActAgent(
        tools=[time_tool],
        llm=openai_llm,
        verbose=True,
    )

    openai_workflow = NatWorkflow(entrypoint=openai_agent)
    print("✅ OpenAI Workflow created")

    # Test the OpenAI workflow
    result = await openai_workflow.prompt("What time is it right now?")
    print(f"\n🤖 OpenAI Response:\n{result}")
else:
    print("⏭️ Skipping OpenAI test (no API key)")


✅ OpenAI Workflow created


[AGENT] Failed to parse agent output after 1 attempts, consider enabling or increasing parse_agent_response_max_retries
Failed to determine whether agent is calling a tool: list index out of range
Traceback (most recent call last):
  File "/Users/spastoriza/Documents/Programming/public/nat-fork/src/nat/agent/react_agent/agent.py", line 252, in conditional_edge
    agent_output = state.agent_scratchpad[-1]
                   ~~~~~~~~~~~~~~~~~~~~~~^^^^
IndexError: list index out of range
[AGENT] Ending graph traversal



🤖 OpenAI Response:
Invalid Format: Missing 'Action:' after 'Thought:'



## 5. Save Configuration


In [9]:
# Save configurations
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

# Save NIM config
nim_config_path = config_dir / "nim_llm_example.yaml"
nim_workflow.save_to_config_file(nim_config_path)
print(f"📄 NIM config saved to: {nim_config_path}")

# Save OpenAI config (if available)
if openai_llm:
    openai_config_path = config_dir / "openai_llm_example.yaml"
    openai_workflow.save_to_config_file(openai_config_path)
    print(f"📄 OpenAI config saved to: {openai_config_path}")


📄 NIM config saved to: configs/nim_llm_example.yaml
📄 OpenAI config saved to: configs/openai_llm_example.yaml


## Common Parameters

| Parameter | Type | Description |
|-----------|------|-------------|
| `model_name` | str | Model identifier |
| `temperature` | float | Randomness (0.0-1.0) |
| `max_tokens` | int | Max response length |
| `top_p` | float | Nucleus sampling (0.0-1.0) |
| `name` | str | Name for config reference |

## CLI Commands

```bash
# Run with NIM config
nat run --config_file configs/nim_llm_example.yaml --input "What time is it?"

# Run with OpenAI config
nat run --config_file configs/openai_llm_example.yaml --input "What time is it?"
```

## Summary

✅ **NimLLM** - NVIDIA NIM models (Llama, Mistral, etc.)  
✅ **OpenAILLM** - OpenAI models (GPT-4, GPT-4o, etc.)  
✅ **Environment variables** - Secure API key handling  

## Next Steps

- **[06_memory.ipynb](./06_memory.ipynb)** - Add memory to your agents
- **[07_retrievers.ipynb](./07_retrievers.ipynb)** - RAG with retrievers
